### Customer Table Transformation ###

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    to_date,
    upper
)
customers_bronze = spark.table(
    "finstream_data_pipeline.bronze.customers"
)
customers_silver = (
    customers_bronze

    # Standardize text fields
    .withColumn(
        "first_name",
        trim(col("first_name"))
    )
    .withColumn(
        "last_name",
        trim(col("last_name"))
    )
    .withColumn(
        "email",
        lower(trim(col("email")))
    )
    .withColumn(
        "country",
        upper(trim(col("country")))
    )
    .withColumn(
        "customer_status",
        lower(trim(col("customer_status")))
    )

    # Ensure date type
    .withColumn(
        "date_of_birth",
        to_date(col("date_of_birth"))
    )

    # Remove ingestion metadata from the curated Silver layer
    .drop(
        "_source_file",
        "_ingestion_timestamp"
    )
)
customers_silver.show(10)
(
    customers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "finstream_data_pipeline.silver.customers"
    )
)

+-----------+----------+------------+--------------------+-------------+-------------+-------+--------------------+--------------------+---------------+
|customer_id|first_name|   last_name|               email|        phone|date_of_birth|country|          created_at|          updated_at|customer_status|
+-----------+----------+------------+--------------------+-------------+-------------+-------+--------------------+--------------------+---------------+
|cust_000001|     Ekaja|       Magar|ayaanmerchant@exa...|   9373225676|   1995-04-26|     IN|2023-07-11 21:37:...|2024-10-07 04:07:...|         active|
|cust_000002|     Manya|Ramachandran|vohrajanuja@examp...|+912634018756|   1967-03-24|     IN|2022-06-13 13:18:...|2025-10-11 14:21:...|       inactive|
|cust_000003|     Janya|          De| qabil80@example.net|  09565337367|   1978-05-20|     IN|2022-07-24 05:58:...|2024-02-29 05:55:...|         active|
|cust_000004|     Anmol|     Narayan|thomasviswanathan...|   4256279569|   1968-06

### Accounts Table Transformations ###

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper
)

accounts_bronze = spark.table(
    "finstream_data_pipeline.bronze.accounts"
)
accounts_silver = (
    accounts_bronze

    .withColumn(
        "account_type",
        lower(trim(col("account_type")))
    )

    .withColumn(
        "account_status",
        lower(trim(col("account_status")))
    )

    .withColumn(
        "currency",
        upper(trim(col("currency")))
    )

    .drop(
        "_source_file",
        "_ingestion_timestamp"
    )
)
accounts_silver.show(10)
(
    accounts_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "finstream_data_pipeline.silver.accounts"
    )
)

+------------+-----------+------------+--------+---------+--------------+--------------------+--------------------+
|  account_id|customer_id|account_type|currency|  balance|account_status|           opened_at|          updated_at|
+------------+-----------+------------+--------+---------+--------------+--------------------+--------------------+
|acc_00000001|cust_006985|    checking|     GBP|412418.31|        active|2021-10-07 14:32:...|2025-04-26 06:53:...|
|acc_00000002|cust_007359|    checking|     USD|481201.01|        active|2024-05-06 12:44:...|2024-10-01 08:31:...|
|acc_00000003|cust_003472|  investment|     INR|139728.59|        active|2024-10-23 12:24:...|2026-06-06 00:13:...|
|acc_00000004|cust_002349|     savings|     USD| 157479.3|      inactive|2021-09-26 04:05:...|2022-05-14 14:06:...|
|acc_00000005|cust_007788|     savings|     GBP|301629.34|      inactive|2023-08-24 12:57:...|2025-08-12 11:11:...|
|acc_00000006|cust_002128|    checking|     EUR|116749.84|        active

### Merchants Table Transformation ###

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper
)

merchants_bronze = spark.table(
    "finstream_data_pipeline.bronze.merchants"
)
merchants_silver = (
    merchants_bronze

    .withColumn(
        "merchant_name",
        trim(col("merchant_name"))
    )

    .withColumn(
        "merchant_category",
        lower(trim(col("merchant_category")))
    )

    .withColumn(
        "country",
        upper(trim(col("country")))
    )

    .withColumn(
        "status",
        lower(trim(col("merchant_status")))
    )

    .drop(
        "_source_file",
        "_ingestion_timestamp"
    )
)
merchants_silver.show(10)
(
    merchants_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "finstream_data_pipeline.silver.merchants"
    )
)

+-----------+--------------------+-----------------+-------+-------------+--------+---------------+--------------------+--------------------+--------+
|merchant_id|       merchant_name|merchant_category|country|         city|currency|merchant_status|          created_at|          updated_at|  status|
+-----------+--------------------+-----------------+-------+-------------+--------+---------------+--------------------+--------------------+--------+
| mer_000001|         Pillay-Sant|      electronics|     AE|     Tirupati|     AED|         active|2024-01-04 19:42:...|2025-01-05 17:16:...|  active|
| mer_000002|Soni, Swaminathan...|        utilities|     DE|Visakhapatnam|     EUR|         active|2021-09-05 22:32:...|2022-06-27 21:53:...|  active|
| mer_000003|         Buch-Thaman|      electronics|     IN|     Thrissur|     INR|       inactive|2024-01-17 11:49:...|2026-08-10 14:09:...|inactive|
| mer_000004|            Kant LLC|             fuel|     US|       Ambala|     USD|         ac

### Exchange_Rates Table Transformation ###

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date
)

exchange_rates_bronze = spark.table(
    "finstream_data_pipeline.bronze.exchange_rates"
)
exchange_rates_silver = (
    exchange_rates_bronze

    .withColumn(
        "from_currency",
        upper(trim(col("base_currency")))
    )

    .withColumn(
        "to_currency",
        upper(trim(col("target_currency")))
    )

    .withColumn(
        "rate_date",
        to_date(col("rate_date"))
    )

    .drop(
        "_source_file",
        "_ingestion_timestamp"
    )
)
exchange_rates_silver.show(10)
(
    exchange_rates_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "finstream_data_pipeline.silver.exchange_rates"
    )
)

+----------------+-------------+---------------+--------+----------+------------------+-------------+-----------+
|exchange_rate_id|base_currency|target_currency|    rate| rate_date|            source|from_currency|to_currency|
+----------------+-------------+---------------+--------+----------+------------------+-------------+-----------+
|     fx_00000001|          INR|            USD|0.011988|2025-08-20|synthetic_provider|          INR|        USD|
|     fx_00000002|          INR|            EUR|0.011003|2025-08-20|synthetic_provider|          INR|        EUR|
|     fx_00000003|          INR|            GBP| 0.00947|2025-08-20|synthetic_provider|          INR|        GBP|
|     fx_00000004|          INR|            AED|0.043985|2025-08-20|synthetic_provider|          INR|        AED|
|     fx_00000005|          INR|            SGD|0.016039|2025-08-20|synthetic_provider|          INR|        SGD|
|     fx_00000006|          USD|            INR| 83.1489|2025-08-20|synthetic_provider| 

### Transactions Table Transformation ###

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    to_timestamp
)

transactions_bronze = spark.table(
    "finstream_data_pipeline.bronze.transactions"
)
transactions_silver = (
    transactions_bronze

    # Standardize categorical fields
    .withColumn(
        "transaction_type",
        lower(trim(col("transaction_type")))
    )

    .withColumn(
        "transaction_status",
        lower(trim(col("transaction_status")))
    )

    .withColumn(
        "payment_method",
        lower(trim(col("payment_method")))
    )

    .withColumn(
        "currency",
        upper(trim(col("currency")))
    )

    # Ensure timestamp is correctly typed
    .withColumn(
        "transaction_timestamp",
        to_timestamp(
            col("transaction_timestamp")
        )
    )

    # Keep amount as a numeric measure
    .withColumn(
        "amount",
        col("amount").cast("decimal(18,2)")
    )

    # Remove Bronze metadata
    .drop(
        "_source_file",
        "_ingestion_timestamp"
    )
)
transactions_silver.show(10)
(
    transactions_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "finstream_data_pipeline.silver.transactions"
    )
)

+--------------------+------------+-----------+---------------------+----------------+---------+--------+------------------+--------------+--------------------+
|      transaction_id|  account_id|merchant_id|transaction_timestamp|transaction_type|   amount|currency|transaction_status|payment_method|         description|
+--------------------+------------+-----------+---------------------+----------------+---------+--------+------------------+--------------+--------------------+
|stream_txn_000000...|acc_00000325| mer_001726| 2026-08-21 15:21:...|        purchase| 22384.13|     INR|            failed| bank_transfer|Incidunt blanditi...|
|stream_txn_000000...|acc_00007203| mer_000971| 2026-08-21 15:21:...|         payment| 42766.64|     AED|         completed|        online|  Nemo aperiam amet.|
|stream_txn_000000...|acc_00008629| mer_000053| 2026-08-21 15:21:...|         payment|   569.38|     USD|         completed|          cash|Aliquid iste quis...|
|stream_txn_000000...|acc_00004940

In [0]:
from pyspark.sql.functions import when

transactions_silver = (
    transactions_silver
    .withColumn(
        "merchant_id_valid",
        when(
            col("transaction_type").isin(
                "payment",
                "purchase"
            ) & col("merchant_id").isNull(),
            False
        ).otherwise(True)
    )
)

In [0]:
transactions_silver.groupBy(
    "merchant_id_valid"
).count().show()

+-----------------+------+
|merchant_id_valid| count|
+-----------------+------+
|             true|100063|
+-----------------+------+



In [0]:
transactions_silver = transactions_silver.drop(
    "merchant_id_valid"
)

### Transactions_Events Table Transformation ###

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    to_timestamp
)

events_bronze = spark.table(
    "finstream_data_pipeline.bronze.transaction_events"
)
events_silver = (
    events_bronze

    .withColumn(
        "event_type",
        lower(trim(col("event_type")))
    )

    .withColumn(
        "event_status",
        lower(trim(col("event_status")))
    )

    .withColumn(
        "failure_reason",
        lower(trim(col("failure_reason")))
    )

    .withColumn(
        "event_timestamp",
        to_timestamp(col("event_timestamp"))
    )

    .drop(
        "_source_file",
        "_ingestion_timestamp"
    )
)
events_silver.show(10)
(
    events_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "finstream_data_pipeline.silver.transaction_events"
    )
)


+--------------+--------------+----------+--------------------+------------+--------------+
|      event_id|transaction_id|event_type|     event_timestamp|event_status|failure_reason|
+--------------+--------------+----------+--------------------+------------+--------------+
|evt_0000000001|txn_0000000001| initiated|2025-10-18 17:09:...|     success|          NULL|
|evt_0000000002|txn_0000000001|authorized|2025-10-18 17:10:...|     success|          NULL|
|evt_0000000003|txn_0000000002| initiated|2026-04-22 16:19:...|     success|          NULL|
|evt_0000000004|txn_0000000002|authorized|2026-04-22 16:20:...|     success|          NULL|
|evt_0000000005|txn_0000000002| completed|2026-04-22 16:23:...|     success|          NULL|
|evt_0000000006|txn_0000000003| initiated|2026-03-18 15:50:...|     success|          NULL|
|evt_0000000007|txn_0000000003|authorized|2026-03-18 15:49:...|     success|          NULL|
|evt_0000000008|txn_0000000003| completed|2026-03-18 15:54:...|     success|    

In [0]:
tables = [
    "customers",
    "accounts",
    "merchants",
    "exchange_rates",
    "transactions",
    "transaction_events"
]

for table in tables:
    print("\n" + "=" * 70)
    print(f"SILVER: {table.upper()}")
    print("=" * 70)

    spark.table(
        f"finstream_data_pipeline.silver.{table}"
    ).printSchema()


SILVER: CUSTOMERS
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- country: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp_ntz (nullable = true)
 |-- customer_status: string (nullable = true)


SILVER: ACCOUNTS
root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- account_status: string (nullable = true)
 |-- opened_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)


SILVER: MERCHANTS
root
 |-- merchant_id: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- country: string (nu